# FACULTY SOLUTION — Week 11: Kessana — the hold-up problem
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['psc_terms', 'reserves', 'exit_and_sunk', 'take_grid', 'comparable_terms', 'worked_example_field']}

In [ ]:

def compute(d):
    p = {x.parameter: x.value for x in d['psc_terms'].itertuples()}; rv = {x.parameter: x.value for x in d['reserves'].itertuples()}
    ex = {x.parameter: x.value for x in d['exit_and_sunk'].itertuples()}
    r = {}
    r['profit_oil'] = p['brent'] + p['kessana_differential'] - p['lifting']
    r['annual_mbbl'] = rv['remaining_mbbl'] / rv['production_years']
    r['annuity_factor'] = (1 - (1 + rv['risk_rate']) ** -rv['production_years']) / rv['risk_rate']
    def pv_stay(take, sunk):   # sunk is accepted and deliberately ignored: it is not a forward cash flow
        return r['profit_oil'] * (1 - take) * r['annual_mbbl'] * r['annuity_factor']
    for s in d['take_grid'].itertuples():
        r['margin_' + s.scenario] = r['profit_oil'] * (1 - s.take)
        r['pv_stay_' + s.scenario] = pv_stay(s.take, ex['sunk_capital_musd_reference_only'])
    r['exit_value'] = ex['exit_value_musd']
    r['stay_minus_exit_demanded'] = r['pv_stay_demanded'] - r['exit_value']
    r['indifference_take'] = 1 - ex['exit_value_musd'] / (r['profit_oil'] * r['annual_mbbl'] * r['annuity_factor'])
    r['sunk_invariant'] = pv_stay(p['demanded_take'], 0) == pv_stay(p['demanded_take'], 1e9)
    ct = d['comparable_terms']['government_take']
    r['comparables_min'] = float(ct.min()); r['comparables_max'] = float(ct.max())
    r['demanded_take'] = p['demanded_take']
    return r


In [ ]:
fx = json.load(open('../fixtures/week11_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')